In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

ModuleNotFoundError: No module named 'seaborn'

In [2]:
# Đường dẫn đến tệp CSV
file_path = "D:/diabetes.csv"

# Bước 1: Đọc dữ liệu
try:
    data = pd.read_csv(file_path)
    print("Dữ liệu đã được đọc thành công!")
except Exception as e:
    print(f"Lỗi khi đọc tệp: {e}")

# Bước 2: Hiển thị thông tin cơ bản
print("\nThông tin tổng quan về dữ liệu:")
print(data.info())

print("\nThống kê mô tả dữ liệu:")
print(data.describe())

In [ ]:
# Bước 3: Xử lý giá trị bị thiếu và giá trị bằng 0
columns_to_replace = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

# Thay thế giá trị bằng 0 bằng giá trị trung bình của từng cột
for col in columns_to_replace:
    data[col] = data[col].replace(0, data[col].mean())

print("\nDữ liệu sau khi xử lý giá trị bằng 0:")
print(data[columns_to_replace].describe())

In [3]:
# Chia dữ liệu thành đặc trưng (X) và nhãn (y)
X = data_cleaned.drop(columns=['Outcome'])
y = data_cleaned['Outcome']

# Chia dữ liệu thành tập huấn luyện và kiểm tra
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

In [4]:
# Chuẩn hóa dữ liệu
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Tính toán lỗi cho các giá trị khác nhau của K
error = []
for i in range(1, 30):
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train_scaled, y_train)  # Sử dụng dữ liệu đã chuẩn hóa
    pred_i = knn.predict(X_test_scaled)  # Sử dụng dữ liệu kiểm tra đã chuẩn hóa
    error.append(np.mean(pred_i != y_test))

# Vẽ biểu đồ biểu diễn Error Rate theo K
plt.figure(figsize=(12, 6))
plt.plot(range(1, 30), error, color='red', linestyle='dashed', marker='o',
         markerfacecolor='blue', markersize=10)
plt.title('Error Rate vs. K Value')
plt.xlabel('K Value')
plt.ylabel('Mean Error')

# In giá trị K có lỗi nhỏ nhất
min_error = min(error)
optimal_k = error.index(min_error) + 1
print(f"Minimum error: {min_error:.4f} at K = {optimal_k}")

# Hiển thị biểu đồ
plt.show()

In [6]:
# Áp dụng kNN
k = 23  # số lượng láng giềng
knn = KNeighborsClassifier(n_neighbors=k)
knn.fit(X_train_scaled, y_train)

# Dự đoán trên tập kiểm tra
y_pred = knn.predict(X_test_scaled)
y_proba = knn.predict_proba(X_test_scaled)[:, 1]

In [ ]:
# Confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)

# Hiển thị confusion matrix dưới dạng biểu đồ màu sắc
plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', cbar=False, xticklabels=['Positive', 'Negative'], yticklabels=['Positive', 'Negative'])
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
# Hiển thị từng chỉ số của báo cáo phân loại
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='weighted')
recall = recall_score(y_test, y_pred, average='weighted')
f1 = f1_score(y_test, y_pred, average='weighted')

print("Model Evaluation:")
print("Accuracy:",(100*accuracy),"%")
print("Precision:",(100*precision),"%")
print("Recall:",(100*recall),"%")
print("F1 Score:",(100*f1),"%")

In [ ]:
# ROC và AUC
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc_score = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc_score:.2f})', color='blue')
plt.plot([0, 1], [0, 1], color='red', linestyle='--')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('ROC Curve')
plt.legend()
plt.grid()
plt.show()
